<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Elegir un Modelo de Mora con Evidencia, No con Suerte 🧪
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 07
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/07_Seleccion_Modelos_Optimizacion_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 07 (Selección de modelos y optimización)** con un problema **nuevo**: una cooperativa quiere predecir la **mora en microcréditos** y necesita decidir entre varios modelos **sin engañarse con un número afortunado**. Trabajarás de forma autónoma, sin soluciones resueltas.

Al terminar serás capaz de:

1. Aplicar el **protocolo de evaluación** (desarrollo / prueba sellada) con una **línea base** y cuantificar la incertidumbre de una métrica con un **intervalo de confianza por *bootstrap*** (cuaderno [00 · Evaluación y selección](../07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/00_Evaluacion_y_Seleccion_de_Modelos.ipynb)).
2. Detectar y evitar la **fuga por grupos** al validar con datos repetidos por cliente (cuaderno [01 · Validación cruzada avanzada](../07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/01_Validacion_Cruzada_Avanzada.ipynb)).
3. Comparar dos modelos con una **prueba *t* pareada con la corrección de Nadeau y Bengio** sobre una CV repetida (cuaderno 00 y 01).
4. Buscar hiperparámetros con **búsqueda aleatoria en escala logarítmica** y revisar la **calibración** de las probabilidades con *Brier* (cuadernos [02 · Random Search y GridSearch](../07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/02_Random_Search_y_GridSearch.ipynb) y 00).

---
## Instrucciones

- Resuelve cada reto **de forma autónoma**: completa las celdas con `# TODO:` y ejecuta la celda de **autoverificación** que le sigue (los `assert` fallan con un mensaje claro hasta que el reto esté bien resuelto).
- No modifiques la celda de preparación ni las celdas de autoverificación.
- Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin (< 1 min) sin errores y sin ningún `# TODO` pendiente.
- Todo usa semillas fijas y datos sintéticos: no necesitas descargar nada.

---
### 📦 Preparación del entorno y de los datos

Ejecuta esta celda sin modificarla. Genera dos conjuntos **sintéticos**: `X`, `y` (1 000 solicitudes de microcrédito, `y = 1` significa **mora**, ≈ 25 %) para los retos 1, 3 y 4; y `Xg`, `yg`, `cliente` (150 clientes con 4 créditos cada uno) para el reto 2, donde las filas **de un mismo cliente no son independientes**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import (train_test_split, KFold, GroupKFold, RepeatedStratifiedKFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy.stats import loguniform
import warnings; warnings.filterwarnings("ignore")

# --- Conjunto 1: solicitudes de microcrédito (retos 1, 3 y 4) ---
rng0 = np.random.default_rng(11)
cols = ["ingreso", "deuda", "antiguedad", "edad", "monto", "plazo", "ahorro", "cuotas_previas", "consultas", "score_ext"]
beta = np.array([1.0, -0.8, 0.7, 0.6, -0.5, 0.4, 0.3, 0.0, 0.0, 0.0])   # las 3 últimas variables son ruido
X_arr = rng0.normal(size=(1000, 10))
y_arr = (rng0.random(1000) < 1 / (1 + np.exp(-(X_arr @ beta - 1.5)))).astype(int)
X = pd.DataFrame(X_arr, columns=cols)
y = pd.Series(y_arr, name="mora")

# --- Conjunto 2: 150 clientes x 4 créditos (reto 2). Cada cliente deja una "huella" en sus variables ---
rng = np.random.default_rng(7)
n_cli, k_cred = 150, 4
z = rng.normal(size=n_cli)                       # riesgo latente del cliente
huella = rng.normal(0, 1.5, size=(n_cli, 8))     # rasgos propios del cliente
cliente = np.repeat(np.arange(n_cli), k_cred)
Xg = huella[cliente] + 0.5 * rng.normal(size=(n_cli * k_cred, 8))
Xg[:, 0] += 0.8 * z[cliente]
yg = (rng.random(n_cli * k_cred) < 1 / (1 + np.exp(-2.0 * z[cliente]))).astype(int)

print("X:", X.shape, "| tasa de mora:", round(y.mean(), 3))
print("Xg:", Xg.shape, "| clientes:", len(np.unique(cliente)), "| tasa de mora:", round(yg.mean(), 3))

---
## Reto 1: Protocolo, Línea Base e Intervalo de Confianza 📏🎯 *(20 puntos)*

**Contexto.** El comité de crédito quiere saber «¿qué tan bueno es el modelo?». Una respuesta honesta incluye una **línea base** y la **incertidumbre** del número.

**Enunciado.**
1. Separa `X`, `y` en `X_dev`, `X_test`, `y_dev`, `y_test` con **25 % de prueba**, `stratify=y` y `random_state=0`. La prueba queda **sellada** hasta el paso 3.
2. Entrena sobre `X_dev` (a) una **línea base** `DummyClassifier(strategy="prior")` y (b) una `LogisticRegression` dentro de un *pipeline* con `StandardScaler`. Calcula `auc_base` y `auc_test` (ROC-AUC) sobre `X_test`.
3. Calcula un **intervalo de confianza *bootstrap* percentil del 95 %** para el AUC del modelo: remuestrea **con reemplazo** los índices de `X_test` **300 veces** (usa `np.random.default_rng(0)`), calcula el AUC de cada remuestreo (descarta los que tengan una sola clase) y guarda los valores en `auc_boot`. Define `ic_inf` e `ic_sup` como los percentiles 2.5 y 97.5.

**Criterios de aceptación.** Partición estratificada con las formas correctas; `auc_base ≈ 0.5`; `auc_test` claramente por encima de la línea base; `ic_inf < auc_test < ic_sup` con un ancho razonable (< 0.15).

In [ ]:
# TODO 1: partición estratificada 75 % / 25 % con random_state=0
X_dev = X_test = y_dev = y_test = None

# TODO 2: línea base (Dummy "prior") y modelo (StandardScaler + LogisticRegression) entrenados SOLO con X_dev
modelo = None
auc_base = None
auc_test = None

# TODO 3: bootstrap percentil del AUC sobre el conjunto de prueba (300 remuestreos)
rng_boot = np.random.default_rng(0)
auc_boot = []
ic_inf = ic_sup = None

print("AUC base:", auc_base, "| AUC modelo:", auc_test, "| IC 95 %:", (ic_inf, ic_sup))

In [ ]:
# ✅ Autoverificación del Reto 1 (no modifiques esta celda)
assert X_dev is not None and y_test is not None, "Reto 1: aún no has creado la partición (X_dev, X_test, y_dev, y_test)."
assert X_dev.shape == (750, 10) and X_test.shape == (250, 10), f"Reto 1: formas inesperadas {X_dev.shape} y {X_test.shape}; ¿usaste test_size=0.25?"
assert abs(y_test.mean() - y.mean()) < 0.02, "Reto 1: la partición debe ser estratificada (la tasa de mora de la prueba debe parecerse a la global)."
assert auc_base is not None and abs(auc_base - 0.5) < 1e-9, "Reto 1: la línea base 'prior' debe dar un AUC de exactamente 0.5."
assert auc_test is not None and auc_test > 0.70, "Reto 1: el modelo debería superar claramente la línea base (AUC > 0.70). ¿Entrenaste con X_dev y evaluaste con X_test?"
assert len(auc_boot) >= 250, "Reto 1: guarda en auc_boot el AUC de cada remuestreo bootstrap (≈ 300 valores)."
assert ic_inf is not None and ic_inf < auc_test < ic_sup, "Reto 1: el AUC puntual debe quedar dentro del intervalo (ic_inf, ic_sup)."
assert 0.0 < ic_sup - ic_inf < 0.15, f"Reto 1: el ancho del intervalo ({ic_sup - ic_inf:.3f}) no es razonable; revisa que remuestreas CON reemplazo y el tamaño de cada remuestreo."
print("✅ Reto 1 superado.")

---
## Reto 2: La Fuga por Grupos en la Validación Cruzada 👥🚨 *(20 puntos)*

**Contexto.** En el conjunto 2 cada cliente aparece en **4 filas**. Si una validación cruzada común reparte las filas al azar, el modelo «reconoce» al cliente por su huella y el puntaje sale **inflado**.

**Enunciado.**
1. Con `RandomForestClassifier(n_estimators=25, random_state=0)` calcula `auc_kfold`: CV con `KFold(5, shuffle=True, random_state=0)` sobre `Xg`, `yg` y `scoring="roc_auc"` (promedio de los 5 pliegues).
2. Calcula `auc_grupos` con `GroupKFold(n_splits=5)` pasando `groups=cliente`.
3. Define `brecha = auc_kfold - auc_grupos`.
4. Implementa `hay_fuga(splitter, X, grupos)`, que devuelva `True` si en **algún pliegue** existe al menos un grupo presente a la vez en entrenamiento y en prueba, y `False` en caso contrario. Guarda `fuga_kfold` y `fuga_grupos` aplicándola a ambos *splitters*.

**Criterios de aceptación.** `fuga_kfold` verdadero y `fuga_grupos` falso; `auc_kfold > auc_grupos` con una brecha de al menos 0.03.

In [ ]:
rf = RandomForestClassifier(n_estimators=25, random_state=0, n_jobs=1)

# TODO 1-3: puntajes de CV con y sin respetar los grupos, y la brecha
auc_kfold = None
auc_grupos = None
brecha = None

def hay_fuga(splitter, X, grupos):
    """True si algún pliegue tiene un mismo grupo en entrenamiento y en prueba."""
    # TODO 4: recorre splitter.split(X, groups=grupos) y compara los conjuntos de grupos de cada lado
    pass

fuga_kfold = None    # TODO: hay_fuga(KFold(...), ...)
fuga_grupos = None   # TODO: hay_fuga(GroupKFold(...), ...)
print("AUC KFold:", auc_kfold, "| AUC GroupKFold:", auc_grupos, "| brecha:", brecha)
print("¿Fuga en KFold?", fuga_kfold, "| ¿Fuga en GroupKFold?", fuga_grupos)

In [ ]:
# ✅ Autoverificación del Reto 2 (no modifiques esta celda)
assert auc_kfold is not None and auc_grupos is not None, "Reto 2: calcula auc_kfold y auc_grupos con cross_val_score."
assert 0.5 < auc_grupos < auc_kfold <= 1.0, "Reto 2: debería cumplirse 0.5 < auc_grupos < auc_kfold. Revisa que GroupKFold reciba groups=cliente."
assert brecha is not None and abs(brecha - (auc_kfold - auc_grupos)) < 1e-12, "Reto 2: brecha debe ser auc_kfold - auc_grupos."
assert brecha > 0.03, f"Reto 2: la brecha ({brecha:.3f}) debería ser > 0.03: la CV común infla el puntaje cuando hay clientes repetidos."
assert hay_fuga(KFold(3, shuffle=True, random_state=1), Xg, cliente) is True, "Reto 2: hay_fuga debe devolver True para KFold aleatorio (hay clientes en ambos lados)."
assert hay_fuga(GroupKFold(n_splits=3), Xg, cliente) is False, "Reto 2: hay_fuga debe devolver False para GroupKFold."
assert fuga_kfold is True and fuga_grupos is False, "Reto 2: guarda fuga_kfold=True y fuga_grupos=False aplicando tu función."
print("✅ Reto 2 superado.")

---
## Reto 3: ¿La Diferencia entre Dos Modelos es Real? ⚖️ *(20 puntos)*

**Contexto.** El analista senior propone un **árbol de decisión** (`max_depth=3`, fácil de explicar al comité); tú prefieres la **regresión logística**. ¿La diferencia observada es real o es ruido de la CV?

**Enunciado.** Sobre `X_dev`, `y_dev`:
1. Define `cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)` y calcula los puntajes **ROC-AUC** de ambos modelos con **el mismo** `cv`: `s_lr` (regresión logística con `StandardScaler`) y `s_arbol` (`DecisionTreeClassifier(max_depth=3, random_state=0)`). Obtienes 15 valores por modelo.
2. Calcula las diferencias pareadas `d = s_lr - s_arbol`, con media $\bar d$ y varianza muestral $\hat\sigma_d^2$ (`ddof=1`).
3. Estadístico ingenuo: $t_{ing} = \bar d \big/ \sqrt{\hat\sigma_d^2 / k}$, con $k = 15$.
4. Estadístico corregido de **Nadeau y Bengio**:
$$t_{corr} = \frac{\bar d}{\sqrt{\left(\frac{1}{k} + \frac{n_{test}}{n_{train}}\right)\hat\sigma_d^2}},\qquad \frac{n_{test}}{n_{train}} = \frac{1}{4}\ \text{(5 pliegues)}.$$
5. Calcula los valores $p$ de dos colas con la distribución $t$ de Student con $k-1$ grados de libertad (`p_ing`, `p_corr`) y define `significativa = p_corr < 0.05`.

**Criterios de aceptación.** 15 valores por modelo; $|t_{corr}| < |t_{ing}|$ y `p_corr >= p_ing` (la corrección es más conservadora); `significativa` es un booleano coherente con `p_corr`.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)

# TODO 1: puntajes ROC-AUC de ambos modelos con el MISMO cv
s_lr = None
s_arbol = None

# TODO 2: diferencias pareadas, su media y su varianza muestral
d = None
k = 15

# TODO 3-4: estadísticos t ingenuo y corregido (Nadeau y Bengio)
t_ing = None
t_corr = None

# TODO 5: valores p de dos colas (t de Student, k-1 g.l.) y decisión al 5 %
p_ing = None
p_corr = None
significativa = None

print("AUC medio LR:", None if s_lr is None else s_lr.mean(), "| AUC medio árbol:", None if s_arbol is None else s_arbol.mean())
print("t ingenuo:", t_ing, "p:", p_ing, "| t corregido:", t_corr, "p:", p_corr, "| ¿significativa?", significativa)

In [ ]:
# ✅ Autoverificación del Reto 3 (no modifiques esta celda)
assert s_lr is not None and s_arbol is not None, "Reto 3: calcula s_lr y s_arbol con cross_val_score."
assert len(s_lr) == 15 and len(s_arbol) == 15, "Reto 3: deben ser 15 puntajes por modelo (5 pliegues x 3 repeticiones); usa el cv dado."
assert s_lr.mean() > s_arbol.mean(), "Reto 3: en estos datos la regresión logística debería superar en promedio al árbol de profundidad 3; revisa los modelos."
assert d is not None and np.allclose(d, s_lr - s_arbol), "Reto 3: d debe ser s_lr - s_arbol."
assert t_ing is not None and t_corr is not None, "Reto 3: calcula t_ing y t_corr."
assert t_ing > 0 and 0 < t_corr < t_ing, "Reto 3: el estadístico corregido debe ser positivo y MENOR que el ingenuo (el factor 1/k + n_test/n_train es mayor que 1/k)."
assert abs(t_ing / t_corr - np.sqrt((1 / 15 + 0.25) / (1 / 15))) < 1e-6, "Reto 3: la razón t_ing/t_corr debe ser sqrt((1/k + 1/4)/(1/k)); revisa el factor de corrección."
assert 0 <= p_ing <= p_corr <= 1, "Reto 3: debe cumplirse 0 <= p_ing <= p_corr <= 1 (dos colas, k-1 grados de libertad)."
assert isinstance(significativa, (bool, np.bool_)) and bool(significativa) == bool(p_corr < 0.05), "Reto 3: significativa debe ser un booleano que refleje p_corr < 0.05."
print("✅ Reto 3 superado.")

---
## Reto 4: Búsqueda Aleatoria en Escala Logarítmica y Calibración 🎲🌡️ *(20 puntos)*

**Contexto.** Elegiste un **SVM con *kernel* RBF**. Sus hiperparámetros `C` y `gamma` varían en **órdenes de magnitud**: se buscan en escala **logarítmica**. Además, el comité usará las probabilidades para fijar tasas de interés, así que deben estar **calibradas**.

**Enunciado.**
1. Crea `busqueda = RandomizedSearchCV(...)` sobre `make_pipeline(StandardScaler(), SVC(random_state=0))` con `C ~ loguniform(1e-2, 1e2)`, `gamma ~ loguniform(1e-3, 1)` (en el *pipeline* los nombres son `svc__C` y `svc__gamma`), `n_iter=12`, `cv=3`, `scoring="roc_auc"`, `random_state=0`. Ajústala **solo con `X_dev`, `y_dev`**.
2. Calibra el mejor modelo: `cal = CalibratedClassifierCV(busqueda.best_estimator_, method="sigmoid", cv=3)` ajustado con `X_dev`, `y_dev`. Obtén `p_cal` = probabilidad de mora sobre `X_test`.
3. Calcula `brier_cal = brier_score_loss(y_test, p_cal)` y la `brier_base` de un modelo constante que predice siempre la tasa de mora de `y_dev`.

**Criterios de aceptación.** 12 candidatos evaluados, con `C` y `gamma` dentro de los rangos y que abarquen varios órdenes de magnitud; `busqueda.best_score_ > 0.75`; `p_cal` en $[0,1]$ y `brier_cal < brier_base`.

In [ ]:
pipe_svm = make_pipeline(StandardScaler(), SVC(random_state=0))

# TODO 1: espacio de búsqueda en escala logarítmica y RandomizedSearchCV (n_iter=12, cv=3, scoring="roc_auc")
espacio = None
busqueda = None   # recuerda llamar a .fit(X_dev, y_dev)

# TODO 2: calibración sigmoide del mejor modelo y probabilidades sobre la prueba
cal = None
p_cal = None

# TODO 3: Brier del modelo calibrado y de la línea base constante
brier_cal = None
brier_base = None
print("Mejores hiperparámetros:", None if busqueda is None else busqueda.best_params_)
print("Brier calibrado:", brier_cal, "| Brier base constante:", brier_base)

In [ ]:
# ✅ Autoverificación del Reto 4 (no modifiques esta celda)
assert busqueda is not None and hasattr(busqueda, "cv_results_"), "Reto 4: crea y AJUSTA busqueda (RandomizedSearchCV.fit)."
cand = busqueda.cv_results_["params"]
assert len(cand) == 12, f"Reto 4: se esperaban 12 candidatos (n_iter=12) y hay {len(cand)}."
Cs = np.array([c["svc__C"] for c in cand]); gs = np.array([c["svc__gamma"] for c in cand])
assert Cs.min() >= 1e-2 and Cs.max() <= 1e2 and gs.min() >= 1e-3 and gs.max() <= 1, "Reto 4: C debe estar en [1e-2, 1e2] y gamma en [1e-3, 1]."
assert Cs.max() / Cs.min() > 50 and gs.max() / gs.min() > 50, "Reto 4: C y gamma deben muestrearse en escala logarítmica (loguniform), abarcando varios órdenes de magnitud."
assert busqueda.best_score_ > 0.75, f"Reto 4: el mejor AUC de CV ({busqueda.best_score_:.3f}) debería ser > 0.75; ¿ajustaste con X_dev, y_dev y scoring='roc_auc'?"
assert p_cal is not None and p_cal.shape == (250,) and p_cal.min() >= 0 and p_cal.max() <= 1, "Reto 4: p_cal debe tener 250 probabilidades en [0, 1] (predict_proba[:, 1] sobre X_test)."
assert brier_base is not None and abs(brier_base - brier_score_loss(y_test, np.full(250, y_dev.mean()))) < 1e-9, "Reto 4: brier_base es el Brier de predecir siempre la tasa de mora de y_dev."
assert brier_cal is not None and brier_cal < brier_base, f"Reto 4: el Brier calibrado ({brier_cal:.3f}) debería ser menor que el de la línea base constante ({brier_base:.3f})."
print("✅ Reto 4 superado.")

---
## Reto 5: Conclusiones 🧾 *(20 puntos)*

Escribe **5 a 8 líneas** interpretando tus resultados **como si se los presentaras al comité de crédito**. Debes referirte, con tus números, a:

- el AUC del modelo frente a la línea base y el **ancho de su intervalo** (Reto 1): ¿qué tan seguro estás del número?
- la **brecha** entre la CV común y la CV por grupos, y por qué ocurre (Reto 2);
- si la ventaja de la regresión logística sobre el árbol es **estadísticamente creíble** una vez aplicada la corrección (Reto 3);
- qué aporta buscar en escala logarítmica y calibrar las probabilidades (Reto 4), y **una limitación** de todo el estudio (por ejemplo, datos sintéticos, una sola partición de prueba, 12 candidatos).

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5 a 8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: protocolo, línea base e IC *bootstrap* correctos | 20 |
| Reto 2: comparación KFold vs. GroupKFold y detector de fuga | 20 |
| Reto 3: prueba *t* pareada con la corrección de Nadeau y Bengio | 20 |
| Reto 4: búsqueda logarítmica, calibración y Brier frente a la base | 20 |
| Reto 5: conclusiones numéricas, honestas y con una limitación | 20 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] No queda ningún `# TODO` ni variable en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen «✅ Reto N superado».
- [ ] *Restart & Run All* corre de principio a fin sin errores.
- [ ] La celda de conclusiones está escrita con tus propias palabras y con números.
- [ ] No modificaste la celda de preparación ni las autoverificaciones.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>